# 16-тарау. Емтихан ұпайын болжаймыз

*«Жасанды интеллект негіздері» кітабына арналған блокнот. Әр ұяшықтың түсіндірмесі кітапта берілген.*

Ұяшықтарды жоғарыдан төмен қарай ретімен іске қосыңыз: **Shift + Enter**.

## 16.1. Жобаның мақсаты

## 16.2. 1-қадам. Деректерді дайындау

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(2024)
n = 200
study = rng.uniform(0, 10, n).round(1)          # дайындық, сағат/күн
sleep = rng.uniform(4, 10, n).round(1)          # ұйқы, сағат
attendance = rng.uniform(50, 100, n).round(0)   # сабаққа қатысу, %
phone = rng.uniform(0, 8, n).round(1)           # телефон, сағат/күн
prev = rng.uniform(40, 100, n).round(0)         # алдыңғы тоқсан бағасы

# «жасырын» формула + кездейсоқ шу
score = (10 + 4 * study + 1.5 * (sleep - 4) + 0.25 * attendance
         - 2 * phone + 0.3 * prev + rng.normal(0, 5, n))
score = score.clip(0, 100).round(0)

df = pd.DataFrame({"сағат": study, "ұйқы": sleep, "қатысу": attendance,
                   "телефон": phone, "алдыңғы": prev, "ұпай": score})
df.to_csv("students.csv", index=False)          # файлға сақтаймыз
print(df.head())

## 16.3. 2-қадам. Деректерді зерттеу

In [ ]:
print(df.describe().round(1))

In [ ]:
print(df.corr()["ұпай"].round(2))

## 16.4. 3-қадам. Модельді оқыту

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

X = df.drop(columns="ұпай")
y = df["ұпай"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

print("Салмақтар (коэффициенттер):")
for name, coef in zip(X.columns, model.coef_):
    print(f"  {name:8s} {coef:+.2f}")
print(f"Ығысу: {model.intercept_:.2f}")

## 16.5. 4-қадам. Модельді түсіндіру

## 16.6. 5-қадам. Бағалау

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

pred = model.predict(X_test)
print(f"MAE: {mean_absolute_error(y_test, pred):.2f} ұпай")
print(f"R²:  {r2_score(y_test, pred):.3f}")

## 16.7. 6-қадам. Қолдану: «Не болады, егер...?»

In [ ]:
aliya = pd.DataFrame([{"сағат": 6, "ұйқы": 8, "қатысу": 95,
                       "телефон": 2, "алдыңғы": 75}])
print(f"Әлияның болжамды ұпайы: {model.predict(aliya)[0]:.1f}")

# Әлия телефонда күніне 6 сағат өткізсе ше?
aliya_phone = aliya.assign(телефон=6)
print(f"Телефон 6 сағат болса: {model.predict(aliya_phone)[0]:.1f}")